In [1]:
%load_ext autoreload
%autoreload 2

### combine meta with vic and ht

In [2]:
"""
This cell does the initial project setup.
If you start a new script or notebook, make sure to copy & paste this part.

A script with this code uses the location of the `.env` file as the anchor for
the whole project (= PROJECT_ROOT). Afterwards, code inside the `src` directory
are available for import.
"""
from pathlib import Path
import sys
from dotenv import load_dotenv, find_dotenv
load_dotenv()
PROJECT_ROOT = Path(find_dotenv()).parent
sys.path.append(str(PROJECT_ROOT.joinpath('src')))

In [ ]:
from utils import olc_client
c = olc_client.connect(verbose=False)

In [4]:
import pandas as pd
from utils.config import (
    CACHE_DIR, DATA_DIR, DATASET, FIG_DIR, HIT_THRE, N_FLOW, PARAMS_DIR, SIDE_CHAR,
)

DATA_DIR.mkdir(parents=True, exist_ok=True)

result_dir = FIG_DIR / 'quan_propagation'
result_dir.mkdir(parents=True, exist_ok=True)

cache_dir = CACHE_DIR / 'quan_propagation'
cache_dir.mkdir(parents=True, exist_ok=True)

# Some setups

## ol types

In [7]:
# cell type table xlsx
oltypes = pd.read_excel(PARAMS_DIR / 'Nern-et-al_SuppTable01_Cell-types-and-counts.xlsx')
oltypes.rename(columns={'cell type':'cell_type', 'main groups':'main_groups'}, inplace=True)
print(oltypes.shape)

oltypes_nonvpn = oltypes[~oltypes['main_groups'].str.contains('VPN')]
oltypes_vpn = oltypes[oltypes['main_groups'].str.contains('VPN')]
oltypes_vcn = oltypes[oltypes['main_groups'].str.contains('VCN')]
oltypes_ol = oltypes[oltypes['main_groups'].str.contains('^ON')]

(778, 6)


## Get meta

In [ ]:
# LOAD meta, this has AN's R7/8, 
meta = pd.read_csv(DATA_DIR / f'{DATASET}_meta.csv')

#  add superclass and class
meta.loc[meta['cell_type'].str.contains('^R7|^R8'), 'superclass'] = 'ol_sensory'
meta.loc[meta['cell_type'].str.contains('^R7|^R8'), 'class'] = 'visual'

print(meta.shape)  

(162084, 11)


In [9]:
# make L1 excitatory so it's an ON cell 
meta.loc[meta.cell_type == 'L1', 'sign'] = 1

In [10]:
# DEBUG
aa = meta.loc[meta['cell_type'].str.contains('^R7|^R8')]
print(aa[~aa.superclass.isin(['ol_sensory'])])
print(aa[~aa['class'].isin(['visual'])])

Empty DataFrame
Columns: [bodyId, instance, cell_type, nt, superclass, subclass, class, coords, sign, side, idx]
Index: []
Empty DataFrame
Columns: [bodyId, instance, cell_type, nt, superclass, subclass, class, coords, sign, side, idx]
Index: []


# vpn and vcbn, ht + vic

In [11]:
# right-side vpn and cb
vp_cb_r = meta[
    ~meta['cell_type'].isin(oltypes['cell_type'].tolist() + ['R7', 'R8', 'R7R8_unclear', 'R7_unclear', 'R8_unclear']) |
    meta['instance'].isin(oltypes[oltypes['main_groups']=='VPN']['instance']) 
]

# # this also works
# meta_cb_vpn = meta[(meta['instance'].isin(oltypes_vpn['instance'])) |
#                    (~meta['cell_type'].isin(oltypes['cell_type'].tolist()) )
#                    ].copy()
# meta_cb_vpn = meta_cb_vpn[meta_cb_vpn['class'] != 'visual']
# meta_cb_vpn.reset_index(drop=True, inplace=True)

# left-side vpn and cb
vp_cb_l = meta[
    ~meta['cell_type'].isin(oltypes['cell_type'].tolist() + ['R7', 'R8',  'R7R8_unclear', 'R7_unclear', 'R8_unclear']) |
    (
        meta['cell_type'].isin(oltypes[oltypes['main_groups']=='VPN']['cell_type']) &
        ~meta['instance'].isin(oltypes[oltypes['main_groups']=='VPN']['instance'])
    )
]

vp_cb_r.shape, vp_cb_l.shape

((63772, 11), (63511, 11))

In [ ]:
# add ht and vic

# load hitting time
ht_r = pd.read_csv(DATA_DIR / f'{DATASET}_r_flow_{N_FLOW}step_{HIT_THRE}thre_hit.csv')
ht_l = pd.read_csv(DATA_DIR / f'{DATASET}_l_flow_{N_FLOW}step_{HIT_THRE}thre_hit.csv')
print(ht_r.shape, ht_l.shape)

# from utils.vic import get_ol_cb_vic_raw
# load vic
vic_r = pd.read_parquet(Path(DATA_DIR, f'{DATASET}_r_ol_cb_vic_raw.parquet'))
vic_l = pd.read_parquet(Path(DATA_DIR, f'{DATASET}_l_ol_cb_vic_raw.parquet'))
print(vic_r.shape, vic_l.shape)

# combine
vp_cb_r = pd.merge(vp_cb_r[['bodyId','cell_type','idx']], ht_r, on='idx', how='left')
vp_cb_l = pd.merge(vp_cb_l[['bodyId','cell_type','idx']], ht_l, on='idx', how='left')
print(vp_cb_r.shape, vp_cb_l.shape)

vp_cb_r = pd.merge(vp_cb_r, vic_r[['bodyId','VIC', 'main_groups', 'region']], on='bodyId', how='inner')
vp_cb_l = pd.merge(vp_cb_l, vic_l[['bodyId','VIC', 'main_groups', 'region']], on='bodyId', how='inner')
print(vp_cb_r.shape, vp_cb_l.shape)

(162084, 3) (162084, 3)
(42413, 5) (42378, 5)
(63772, 5) (63511, 5)
(41757, 8) (41609, 8)


In [ ]:
# save to cache
vp_cb_r.to_pickle(Path(DATA_DIR, 'vp_cb_hit_vic_r.p'))
vp_cb_l.to_pickle(Path(DATA_DIR, 'vp_cb_hit_vic_l.p'))

# # load
vp_cb_r = pd.read_pickle(Path(DATA_DIR, 'vp_cb_hit_vic_r.p'))
vp_cb_l = pd.read_pickle(Path(DATA_DIR, 'vp_cb_hit_vic_l.p'))

# End